# CS2 EXP-5 — HEFT


## 1. Dependencies


In [1]:
import sys
import subprocess

subprocess.run([
    sys.executable, "-m", "pip", "install", "-q", "-U",
    "--extra-index-url", "https://download.pytorch.org/whl/cu121",
    "torch==2.5.1",
    "torchvision",
    "torchaudio",
    "transformers<4.49.0",  # Avoids the CVE check enforcing PyTorch 2.6
    "peft<0.14.0",
    "accelerate",
    "numpy<2.1.0",
    "pandas",
    "scikit-learn",
    "matplotlib",
    "pyarrow",
    "joblib",
    "tqdm",
    "psutil",
], check=True)

print("Dependencies installed successfully for CUDA 12.1 driver!")

Dependencies installed successfully for CUDA 12.1 driver!


In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import torch

assert torch.cuda.is_available(), "CUDA is not available!"
assert torch.cuda.device_count() == 1, f"Expected 1 GPU, but PyTorch sees {torch.cuda.device_count()}"

DEVICE = "cuda:0"

print(f"Locked to single GPU: {torch.cuda.get_device_name(0)}")
print(f"Total Visible GPUs in PyTorch: {torch.cuda.device_count()}")

Locked to single GPU: NVIDIA A100-SXM4-80GB
Total Visible GPUs in PyTorch: 1


## 1.5 Settings


In [3]:
import os
from pathlib import Path

REPO_URL = "https://github.com/EnomisLP/DiverseVul--IS-Project.git"
REPO_BRANCH = "simo"

WORKSPACE_ROOT = Path.cwd()

REPO_ROOT = WORKSPACE_ROOT / "DiverseVul--IS-Project"
PROJECT_DIR = REPO_ROOT / "vuln-detection"
SRC_DIR = PROJECT_DIR / "src"

DATA_ROOT = WORKSPACE_ROOT / "IntelligentSystemProject" / "VulnerabilityDetectionData"

PROCESSED_DIR = DATA_ROOT / "processed"
MANIFEST_ROOT = DATA_ROOT / "manifests"
OUTPUT_ROOT = DATA_ROOT / "outputs"

SPLIT_ID = "cs1_project_holdout20_innercv_v1"

NORMALIZED_PARQUET = (
    PROCESSED_DIR
    / "rdiversevul_cs1_normalized_plus_abstracted_v1.parquet"
)

OUTER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / SPLIT_ID
    / "outer_holdout"
    / "cs1_outer_project_holdout_manifest.parquet"
)

INNER_MANIFEST_PATH = (
    MANIFEST_ROOT
    / SPLIT_ID
    / "inner_cv"
    / "cs1_project_grouped_5fold_manifest.parquet"
)

EXP5_OUTPUT_DIR = (
    OUTPUT_ROOT
    / "case_study_2"
    / "exp5_graphcodebert_lora_v1"
)

EXP5_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

HF_CACHE_DIR = WORKSPACE_ROOT / "IntelligentSystemProject" / "hf_cache"
os.environ["HF_TOKEN"] = "secret"
RANK_GRID = (8, 16, 32)
EPOCHS = 4
SEARCH_EPOCHS = 2
SEARCH_N_SPLITS = 5

TRAIN_BATCH_SIZE = 32
GRAD_ACCUM_STEPS = 2
EVAL_BATCH_SIZE = 64
NUM_WORKERS = 8

STORAGE_CAP_GB = 60

RUN_SMOKE_TEST = True
RUN_NESTED_OFFICIAL = True
RUN_CANONICAL_RETRAIN = True
RUN_HOLDOUT_EVAL = True

print("Settings loaded.")
print(f"Workspace: {WORKSPACE_ROOT}")
print(f"Repository: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
print(f"Output root: {OUTPUT_ROOT}")
print(f"Hugging Face cache: {HF_CACHE_DIR}")
print(f"Device: {DEVICE}")

Settings loaded.
Workspace: /workspace
Repository: /workspace/DiverseVul--IS-Project
Data root: /workspace/IntelligentSystemProject/VulnerabilityDetectionData
Output root: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/outputs
Hugging Face cache: /workspace/IntelligentSystemProject/hf_cache
Device: cuda:0


## 2. Clone the repository


In [4]:
import urllib.request
import zipfile
import shutil
from pathlib import Path

if REPO_ROOT.exists():
    print(f"Removing existing repository at {REPO_ROOT}...")
    shutil.rmtree(REPO_ROOT)

print(f"Downloading repository (branch: {REPO_BRANCH}) without git...")

clean_url = REPO_URL.removesuffix(".git")
zip_url = f"{clean_url}/archive/refs/heads/{REPO_BRANCH}.zip"
zip_path = Path.cwd() / "repo_temp.zip"

urllib.request.urlretrieve(zip_url, zip_path)

print("Extracting files...")
with zipfile.ZipFile(zip_path, "r") as zip_ref:
    zip_ref.extractall(Path.cwd())

repo_name = clean_url.split("/")[-1]
extracted_folder = Path.cwd() / f"{repo_name}-{REPO_BRANCH}"
if extracted_folder.exists():
    extracted_folder.rename(REPO_ROOT)

zip_path.unlink()
print("Repository setup complete!")

Removing existing repository at /workspace/DiverseVul--IS-Project...
Extracting files...
Repository setup complete!


## 3. Verify GPU, RAM, and storage budget


In [5]:
import shutil
import psutil
import torch

if not torch.cuda.is_available():
    raise RuntimeError("EXP-5 requires a GPU runtime for LoRA fine-tuning.")

assert DEVICE == "cuda:0"

print("CUDA device:", torch.cuda.get_device_name(0))
print("bfloat16 supported:", torch.cuda.is_bf16_supported())
print("Total VRAM: %.2f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))

print("CPU cores:", psutil.cpu_count(logical=True))
print("System RAM: %.1f GB" % (psutil.virtual_memory().total / 1e9))

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage already exceeds the {STORAGE_CAP_GB} GB storage cap for this machine.")

CUDA device: NVIDIA A100-SXM4-80GB
bfloat16 supported: True
Total VRAM: 84.99 GB
CPU cores: 112
System RAM: 2164.0 GB
Disk usage at /workspace: 5129.5 GB used / 5714.2 GB total (296.6 GB free)


## 4. Data availability check


In [6]:
required_data_files = {
    "normalized parquet": NORMALIZED_PARQUET,
    "outer holdout manifest": OUTER_MANIFEST_PATH,
    "inner CV manifest": INNER_MANIFEST_PATH,
}

missing = {name: path for name, path in required_data_files.items() if not path.is_file()}

if missing:
    print("Missing required data files:")
    for name, path in missing.items():
        print(f"  - {name}: {path}")
    print(
        "\nThese files are produced by the Case Study 1 pipeline (normalization_v3.py + "
        "split_manifest.py) and were previously synced through Google Drive.\n"
        f"Keep an eye on the {STORAGE_CAP_GB} GB storage cap."
    )
    raise FileNotFoundError("Required processed data/manifests are missing; see instructions above.")
else:
    for name, path in required_data_files.items():
        size_mb = path.stat().st_size / 1e6
        print(f"Found {name}: {path} ({size_mb:.1f} MB)")

Found normalized parquet: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/processed/rdiversevul_cs1_normalized_plus_abstracted_v1.parquet (210.7 MB)
Found outer holdout manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/outer_holdout/cs1_outer_project_holdout_manifest.parquet (1.4 MB)
Found inner CV manifest: /workspace/IntelligentSystemProject/VulnerabilityDetectionData/manifests/cs1_project_holdout20_innercv_v1/inner_cv/cs1_project_grouped_5fold_manifest.parquet (1.2 MB)


## 5. Write case_study_2 source files


In [7]:
(SRC_DIR / "case_study_2/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/__init__.py").write_text('')
print("Wrote", "case_study_2/__init__.py")

Wrote case_study_2/__init__.py


In [8]:
(SRC_DIR / "case_study_2/models.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/models.py").write_text('''from __future__ import annotations

import os
from pathlib import Path
from typing import Optional, Dict, Any, List

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer


DEFAULT_CODE_MODEL = "microsoft/graphcodebert-base"
DEFAULT_CODE_TOKENIZER = "microsoft/graphcodebert-base"


def configure_huggingface_cache(hf_cache_dir: Optional[str] = None) -> None:
    if hf_cache_dir:
        hf_cache_dir = str(hf_cache_dir)
        os.environ.setdefault("HF_HOME", hf_cache_dir)
        os.environ.setdefault("HUGGINGFACE_HUB_CACHE", str(Path(hf_cache_dir) / "hub"))
    os.environ.setdefault("HF_HUB_DISABLE_XET", "1")
    os.environ.setdefault("HF_HUB_ENABLE_HF_TRANSFER", "0")
    os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
    os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "120")
    os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")


def _dtype_from_policy(dtype_policy: str, device: str) -> Optional[torch.dtype]:
    dtype_policy = (dtype_policy or "auto").lower()
    device = str(device)
    if dtype_policy == "float16":
        return torch.float16 if device == "cuda" else torch.float32
    if dtype_policy == "bfloat16":
        return torch.bfloat16 if device == "cuda" and torch.cuda.is_bf16_supported() else torch.float32
    if dtype_policy == "float32":
        return torch.float32
    if dtype_policy == "auto":
        if device == "cuda" and torch.cuda.is_bf16_supported():
            return torch.bfloat16
        if device == "cuda":
            return torch.float32
        return torch.float32
    raise ValueError(f"Unknown dtype_policy: {dtype_policy}")


def load_code_tokenizer(
    tokenizer_name: str = DEFAULT_CODE_TOKENIZER,
    hf_cache_dir: Optional[str] = None,
):
    configure_huggingface_cache(hf_cache_dir)
    return AutoTokenizer.from_pretrained(
        tokenizer_name,
        use_fast=True,
        cache_dir=hf_cache_dir,
    )


def load_code_encoder(
    model_name: str = DEFAULT_CODE_MODEL,
    dtype_policy: str = "auto",
    device: Optional[str] = None,
    freeze: bool = True,
    hf_cache_dir: Optional[str] = None,
) -> nn.Module:
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    configure_huggingface_cache(hf_cache_dir)
    dtype = _dtype_from_policy(dtype_policy, device)

    kwargs: Dict[str, Any] = {"cache_dir": hf_cache_dir}
    if dtype is not None:
        kwargs["torch_dtype"] = dtype

    model = AutoModel.from_pretrained(model_name, **kwargs)
    model.to(device)

    if freeze:
        for param in model.parameters():
            param.requires_grad = False
        model.eval()

    return model


def mean_pool_last_hidden(last_hidden_state: torch.Tensor, attention_mask: torch.Tensor) -> torch.Tensor:
    mask = attention_mask.unsqueeze(-1).to(last_hidden_state.dtype)
    summed = (last_hidden_state * mask).sum(dim=1)
    denom = mask.sum(dim=1).clamp(min=1.0)
    return summed / denom


def cls_pool_last_hidden(last_hidden_state: torch.Tensor) -> torch.Tensor:
    return last_hidden_state[:, 0, :]


class CodeSequenceClassifier(nn.Module):
    def __init__(
        self,
        model_name: str = DEFAULT_CODE_MODEL,
        num_labels: int = 1,
        freeze_backbone: bool = False,
        pooling: str = "mean",
        dtype_policy: str = "auto",
        hf_cache_dir: Optional[str] = None,
    ) -> None:
        super().__init__()
        device = "cuda" if torch.cuda.is_available() else "cpu"
        self.backbone = load_code_encoder(
            model_name=model_name,
            dtype_policy=dtype_policy,
            device=device,
            freeze=freeze_backbone,
            hf_cache_dir=hf_cache_dir,
        )
        hidden_size = int(self.backbone.config.hidden_size)
        self.classification_head = nn.Linear(hidden_size, num_labels)
        self.pooling = pooling

    def forward(self, input_ids: torch.Tensor, attention_mask: torch.Tensor, **kwargs: Any) -> torch.Tensor:
        outputs = self.backbone(input_ids=input_ids, attention_mask=attention_mask, **kwargs)
        hidden = outputs.last_hidden_state
        if self.pooling == "cls":
            pooled = cls_pool_last_hidden(hidden)
        else:
            pooled = mean_pool_last_hidden(hidden, attention_mask)
        logits = self.classification_head(pooled)
        return logits.squeeze(-1)


def count_trainable_parameters(model: nn.Module) -> Dict[str, int]:
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total = sum(p.numel() for p in model.parameters())
    return {
        "trainable_parameters": int(trainable),
        "total_parameters": int(total),
        "trainable_percent": float(100.0 * trainable / max(total, 1)),
    }


def infer_lora_target_modules(model: nn.Module) -> List[str]:
    module_names = [name for name, _ in model.named_modules()]
    candidate_sets = [
        ["qkv"],
        ["q_proj", "v_proj"],
        ["query", "value"],
        ["in_proj"],
    ]
    for candidates in candidate_sets:
        if all(any(name.endswith(candidate) or f".{candidate}" in name for name in module_names) for candidate in candidates):
            return candidates
    return ["query", "value"]


def create_lora_sequence_classifier(
    model_name: str = DEFAULT_CODE_MODEL,
    rank: int = 8,
    lora_alpha: int = 16,
    lora_dropout: float = 0.05,
    pooling: str = "mean",
    dtype_policy: str = "auto",
    hf_cache_dir: Optional[str] = None,
):
    from peft import LoraConfig, get_peft_model

    base = CodeSequenceClassifier(
        model_name=model_name,
        freeze_backbone=False,
        pooling=pooling,
        dtype_policy=dtype_policy,
        hf_cache_dir=hf_cache_dir,
    )
    target_modules = infer_lora_target_modules(base)
    config = LoraConfig(
        r=rank,
        lora_alpha=lora_alpha,
        target_modules=target_modules,
        lora_dropout=lora_dropout,
        bias="none",
        task_type="FEATURE_EXTRACTION",
    )
    return get_peft_model(base, config)


def get_lora_model(model_name: str = DEFAULT_CODE_MODEL, rank: int = 8, lora_alpha: int = 16, pooling: str = "mean"):
    return create_lora_sequence_classifier(
        model_name=model_name,
        rank=rank,
        lora_alpha=lora_alpha,
        pooling=pooling,
    )
''')
print("Wrote", "case_study_2/models.py")

Wrote case_study_2/models.py


In [9]:
(SRC_DIR / "case_study_2/data_loader.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/data_loader.py").write_text('''from __future__ import annotations

from dataclasses import dataclass
from typing import Optional, Dict, Any, List, Tuple

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader


EMPTY_CODE_SENTINEL = "EMPTY_CODE_SAMPLE"


class CodeTextDataset(Dataset):
    def __init__(
        self,
        dataframe: pd.DataFrame,
        code_column: str = "normalized_code",
        label_column: str = "label",
        source_id_column: str = "source_row_id",
        project_column: str = "project",
    ) -> None:
        self.df = dataframe.copy().reset_index(drop=True)
        self.code_column = code_column
        self.label_column = label_column
        self.source_id_column = source_id_column
        self.project_column = project_column

        self.df[self.code_column] = self.df[self.code_column].fillna("").astype(str)
        empty_mask = self.df[self.code_column].str.strip().eq("")
        if empty_mask.any():
            self.df.loc[empty_mask, self.code_column] = EMPTY_CODE_SENTINEL

    def __len__(self) -> int:
        return int(len(self.df))

    def __getitem__(self, idx: int) -> Dict[str, Any]:
        row = self.df.iloc[idx]
        return {
            "code": str(row[self.code_column]),
            "label": int(row[self.label_column]),
            "source_row_id": int(row[self.source_id_column]),
            "project": str(row[self.project_column]),
        }


@dataclass
class TransformerBatchCollator:
    tokenizer: Any
    max_length: int = 512
    pad_to_multiple_of: Optional[int] = 8

    def __call__(self, features: List[Dict[str, Any]]) -> Dict[str, Any]:
        texts = [feature["code"] for feature in features]
        enc = self.tokenizer(
            texts,
            truncation=True,
            max_length=self.max_length,
            padding=True,
            pad_to_multiple_of=self.pad_to_multiple_of,
            return_tensors="pt",
        )

        labels = torch.tensor([feature["label"] for feature in features], dtype=torch.float32)
        source_row_ids = torch.tensor([feature["source_row_id"] for feature in features], dtype=torch.long)
        projects = [feature["project"] for feature in features]

        enc["labels"] = labels
        enc["label"] = labels
        enc["source_row_id"] = source_row_ids
        enc["project"] = projects
        return enc


def create_dataloader(
    dataframe: pd.DataFrame,
    tokenizer: Any,
    batch_size: int = 16,
    max_length: int = 512,
    shuffle: bool = False,
    code_column: str = "normalized_code",
    label_column: str = "label",
    source_id_column: str = "source_row_id",
    project_column: str = "project",
    num_workers: int = 0,
) -> DataLoader:
    dataset = CodeTextDataset(
        dataframe=dataframe,
        code_column=code_column,
        label_column=label_column,
        source_id_column=source_id_column,
        project_column=project_column,
    )
    collator = TransformerBatchCollator(
        tokenizer=tokenizer,
        max_length=max_length,
        pad_to_multiple_of=8 if torch.cuda.is_available() else None,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=False,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=collator,
    )


def get_pos_weight(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:
    y = dataframe[label_column].astype(int).values
    neg = int((y == 0).sum())
    pos = int((y == 1).sum())
    if pos == 0:
        return torch.tensor([1.0], dtype=torch.float32)
    return torch.tensor([neg / pos], dtype=torch.float32)


def get_class_weights(dataframe: pd.DataFrame, label_column: str = "label") -> torch.Tensor:
    return get_pos_weight(dataframe, label_column=label_column)
''')
print("Wrote", "case_study_2/data_loader.py")

Wrote case_study_2/data_loader.py


In [10]:
(SRC_DIR / "case_study_2/exp5/__init__.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp5/__init__.py").write_text('')
print("Wrote", "case_study_2/exp5/__init__.py")

Wrote case_study_2/exp5/__init__.py


In [11]:
(SRC_DIR / "case_study_2/exp5/exp5_lora.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp5/exp5_lora.py").write_text('''from __future__ import annotations

import gc
import time
import torch
import torch.nn as nn
from torch.optim import AdamW
import numpy as np

from case_study_2.data_loader import create_dataloader, get_class_weights
from case_study_2.models import get_lora_model, count_trainable_parameters, DEFAULT_CODE_MODEL


def train_lora_model(
    train_df,
    val_df,
    tokenizer,
    rank,
    epochs=3,
    batch_size=16,
    grad_accum_steps=2,
    eval_batch_size=32,
    num_workers=2,
    device="cuda",
    hf_cache_dir=None,
    code_column="normalized_code",
    max_length=512,
    verbose=True,
    log_every_steps=50,
    log_prefix="",
):
    train_loader = create_dataloader(
        train_df, tokenizer, batch_size=batch_size, max_length=max_length,
        shuffle=True, num_workers=num_workers, code_column=code_column,
    )
    val_loader = create_dataloader(
        val_df, tokenizer, batch_size=eval_batch_size, max_length=max_length,
        shuffle=False, num_workers=num_workers, code_column=code_column,
    )

    model = get_lora_model(model_name=DEFAULT_CODE_MODEL, rank=rank, lora_alpha=16).to(device)

    is_cuda = (device == "cuda") or (hasattr(device, "type") and device.type == "cuda")

    total_steps_per_epoch = -(-len(train_df) // batch_size)

    if verbose:
        stats = count_trainable_parameters(model)
        print(
            f"{log_prefix}[exp5-lora] rank={rank} | train_rows={len(train_df)} | val_rows={len(val_df)} | "
            f"batch_size={batch_size} | grad_accum={grad_accum_steps} | steps/epoch={total_steps_per_epoch} | "
            f"trainable={stats['trainable_parameters']:,} ({stats['trainable_percent']:.3f}%) | "
            f"total={stats['total_parameters']:,}"
        )
        if is_cuda:
            print(f"{log_prefix}[exp5-lora] VRAM after model load: {torch.cuda.memory_allocated()/1e9:.2f} GB")

    pos_weight = get_class_weights(train_df).to(device)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = AdamW(model.parameters(), lr=2e-4)

    t0 = time.time()

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        n_steps = 0
        epoch_t0 = time.time()
        optimizer.zero_grad()

        for step, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            labels = batch["label"].to(device, non_blocking=True)

            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels) / grad_accum_steps

            loss.backward()
            epoch_loss += loss.item() * grad_accum_steps
            n_steps += 1

            if (step + 1) % grad_accum_steps == 0:
                optimizer.step()
                optimizer.zero_grad()

            if verbose and log_every_steps and (step + 1) % log_every_steps == 0:
                elapsed_min = (time.time() - epoch_t0) / 60
                steps_left = total_steps_per_epoch - (step + 1)
                rate = (step + 1) / max(elapsed_min, 1e-6)
                eta_min = steps_left / max(rate, 1e-6)
                print(
                    f"{log_prefix}[exp5-lora] epoch {epoch+1}/{epochs} step {step+1}/{total_steps_per_epoch} | "
                    f"avg_loss_so_far={epoch_loss/max(n_steps,1):.4f} | "
                    f"elapsed={elapsed_min:.1f} min | ETA epoch ~{eta_min:.1f} min"
                )

        optimizer.step()
        optimizer.zero_grad()

        if verbose:
            elapsed_min = (time.time() - t0) / 60
            epoch_min = (time.time() - epoch_t0) / 60
            peak_vram = torch.cuda.max_memory_allocated() / 1e9 if is_cuda else 0.0
            print(
                f"{log_prefix}[exp5-lora] epoch {epoch+1}/{epochs} done | avg_loss={epoch_loss/max(n_steps,1):.4f} "
                f"| epoch_time={epoch_min:.1f} min | total_elapsed={elapsed_min:.1f} min | peak_VRAM={peak_vram:.2f} GB"
            )

    if verbose:
        print(f"{log_prefix}[exp5-lora] training done, scoring validation set ({len(val_df)} rows)...")

    model.eval()
    all_scores = []
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device, non_blocking=True)
            attention_mask = batch["attention_mask"].to(device, non_blocking=True)
            with torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16):
                logits = model(input_ids, attention_mask)
                scores = torch.sigmoid(logits)
            all_scores.extend(scores.float().cpu().numpy())

    del train_loader, val_loader, criterion, optimizer
    if is_cuda:
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    return np.array(all_scores), model


def train_lora_model_safe(*args, max_retries=2, **kwargs):
    batch_size = kwargs.pop("batch_size", 16)
    grad_accum_steps = kwargs.pop("grad_accum_steps", 2)

    attempt = 0
    while True:
        try:
            return train_lora_model(
                *args, batch_size=batch_size, grad_accum_steps=grad_accum_steps, **kwargs
            )
        except torch.cuda.OutOfMemoryError:
            attempt += 1
            gc.collect()
            torch.cuda.empty_cache()
            if attempt > max_retries or batch_size <= 2:
                raise
            new_batch_size = max(2, batch_size // 2)
            new_grad_accum_steps = grad_accum_steps * max(1, batch_size // new_batch_size)
            print(
                f"[exp5-lora] CUDA OOM at batch_size={batch_size}; retrying "
                f"(attempt {attempt}/{max_retries}) with batch_size={new_batch_size}, "
                f"grad_accum_steps={new_grad_accum_steps} (effective batch size unchanged)."
            )
            batch_size, grad_accum_steps = new_batch_size, new_grad_accum_steps
''')
print("Wrote", "case_study_2/exp5/exp5_lora.py")

Wrote case_study_2/exp5/exp5_lora.py


In [12]:
(SRC_DIR / "case_study_2/exp5/exp5_nested_rank.py").parent.mkdir(parents=True, exist_ok=True)
(SRC_DIR / "case_study_2/exp5/exp5_nested_rank.py").write_text('''from __future__ import annotations

import gc
import json
import time
from dataclasses import dataclass, asdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

import joblib
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, precision_recall_curve, confusion_matrix
import matplotlib.pyplot as plt

from case_study_2.data_loader import create_dataloader, get_class_weights
from case_study_2.models import configure_huggingface_cache, load_code_tokenizer, DEFAULT_CODE_TOKENIZER
from case_study_2.exp5.exp5_lora import train_lora_model_safe
from case_study_1 import split_manifest
from case_study_1 import evaluation
from case_study_1.evaluation import EvaluationConfig
from case_study_1.confidence_intervals import bootstrap_metric_ci, format_ci_report


EXP5_VERSION = "cs2-exp5-graphcodebert-lora-v1"


@dataclass(frozen=True)
class Exp5Config:
    experiment_name: str = "cs2_exp5_graphcodebert_lora"

    code_column: str = "normalized_code"
    source_id_column: str = "source_row_id"
    label_column: str = "label"
    project_column: str = "project"
    fold_column: str = "fold"

    hf_cache_dir: Optional[str] = None
    max_length: int = 512
    train_batch_size: int = 16
    grad_accum_steps: int = 2
    epochs: int = 3

    rank_grid: Tuple[int, ...] = (8, 16)
    inner_n_splits: int = 3
    inner_random_state: int = 20260707
    decision_threshold: float = 0.50

    search_epochs: int = 1
    search_n_splits: int = 4

    num_workers: int = 6
    eval_batch_size: int = 64

    n_splits: int = 5
    random_state: int = 42
    verbose: bool = True


def _checkpoint_paths(output_dir: Path, outer_fold_id: int) -> Dict[str, Path]:
    root = output_dir / "checkpoints"
    root.mkdir(parents=True, exist_ok=True)
    prefix = f"outer_fold_{outer_fold_id}"
    return {
        "predictions": root / f"{prefix}_predictions.parquet",
        "selected": root / f"{prefix}_selected_rank.json",
        "training": root / f"{prefix}_outer_training.json",
    }


def _search_checkpoint_path(output_dir: Path, outer_fold_id: int) -> Path:
    root = output_dir / "checkpoints"
    root.mkdir(parents=True, exist_ok=True)
    return root / f"outer_fold_{outer_fold_id}_search_progress.json"


def _load_search_checkpoint(output_dir: Path, outer_fold_id: int) -> Dict[str, float]:
    path = _search_checkpoint_path(output_dir, outer_fold_id)
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as f:
        raw = json.load(f)
    return {int(k): float(v) for k, v in raw.items()}


def _save_search_checkpoint(output_dir: Path, outer_fold_id: int, rank_performance: Dict[int, float]) -> None:
    path = _search_checkpoint_path(output_dir, outer_fold_id)
    with path.open("w", encoding="utf-8") as f:
        json.dump({str(k): v for k, v in rank_performance.items()}, f, indent=2)


def _write_outer_checkpoint(output_dir: Path, outer_fold_id: int, predictions: pd.DataFrame, selected: dict, training: dict) -> None:
    paths = _checkpoint_paths(output_dir, outer_fold_id)
    predictions.to_parquet(paths["predictions"], index=False)
    with paths["selected"].open("w", encoding="utf-8") as f:
        json.dump(selected, f, indent=2, default=str)
    with paths["training"].open("w", encoding="utf-8") as f:
        json.dump(training, f, indent=2, default=str)


def _load_outer_checkpoint(output_dir: Path, outer_fold_id: int) -> Optional[dict]:
    paths = _checkpoint_paths(output_dir, outer_fold_id)
    if not all(p.exists() for p in paths.values()):
        return None
    with paths["selected"].open("r", encoding="utf-8") as f:
        selected = json.load(f)
    with paths["training"].open("r", encoding="utf-8") as f:
        training = json.load(f)
    return {
        "predictions": pd.read_parquet(paths["predictions"]),
        "selected": selected,
        "training": training,
    }


def _update_run_state(state_path: Path, completed_folds, status: str) -> None:
    state = {
        "status": status,
        "updated_utc": datetime.now(timezone.utc).isoformat(),
        "completed_outer_folds": sorted(int(f) for f in completed_folds),
    }
    with state_path.open("w", encoding="utf-8") as f:
        json.dump(state, f, indent=2)


def run_exp5_nested_rank(
    development_frame: pd.DataFrame,
    development_manifest: pd.DataFrame,
    config: Exp5Config,
    output_dir: Path,
    resume: bool = True,
    additional_metadata: Optional[Dict[str, Any]] = None,
) -> Dict[str, Any]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if device.type != "cuda":
        raise RuntimeError("EXP-5 LoRA fine-tuning requires a CUDA device.")

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    state_path = output_dir / "exp5_nested_run_state.json"

    configure_huggingface_cache(config.hf_cache_dir)
    tokenizer = load_code_tokenizer(DEFAULT_CODE_TOKENIZER, hf_cache_dir=config.hf_cache_dir)

    fold_ids = sorted(development_manifest[config.fold_column].unique().tolist())
    print(f"[nested] Starting EXP-5 nested rank search over {len(fold_ids)} outer folds, rank_grid={config.rank_grid}")
    print(f"[nested] Search phase: {config.search_epochs} epoch(s), single held-out split | Refit phase: {config.epochs} epoch(s), full training")

    oof_parts = []
    selected_rows = []
    outer_training_rows = []
    completed_folds = []

    t0 = time.time()

    for outer_fold_id in fold_ids:
        checkpoint = _load_outer_checkpoint(output_dir, outer_fold_id) if resume else None
        if checkpoint is not None:
            oof_parts.append(checkpoint["predictions"])
            selected_rows.append(checkpoint["selected"])
            outer_training_rows.append(checkpoint["training"])
            completed_folds.append(outer_fold_id)
            if config.verbose:
                print(f"[nested] Outer fold {outer_fold_id}: loaded from checkpoint, skipping.")
            continue

        fold_t0 = time.time()
        print(f"\\n=================== OUTER FOLD {outer_fold_id} ({len(completed_folds)+1}/{len(fold_ids)}) ===================")

        outer_train_ids = development_manifest.loc[
            development_manifest[config.fold_column] != outer_fold_id, config.source_id_column
        ]
        outer_val_ids = development_manifest.loc[
            development_manifest[config.fold_column] == outer_fold_id, config.source_id_column
        ]
        outer_train_df = development_frame[development_frame[config.source_id_column].isin(outer_train_ids)].reset_index(drop=True)
        outer_val_df = development_frame[development_frame[config.source_id_column].isin(outer_val_ids)].reset_index(drop=True)
        print(f"[nested] outer_train={len(outer_train_df)} rows | outer_val={len(outer_val_df)} rows")

        search_split_config = split_manifest.SplitConfig(
            n_splits=config.search_n_splits,
            random_state=config.inner_random_state,
            shuffle=True,
            source_id_column=config.source_id_column,
            label_column=config.label_column,
            group_column=config.project_column,
        )
        search_manifest = split_manifest.create_project_grouped_manifest(
            outer_train_df[[config.source_id_column, config.label_column, config.project_column]],
            config=search_split_config,
        )
        search_train_ids = search_manifest.loc[search_manifest["fold"] != 0, config.source_id_column]
        search_val_ids = search_manifest.loc[search_manifest["fold"] == 0, config.source_id_column]
        search_train_df = outer_train_df[outer_train_df[config.source_id_column].isin(search_train_ids)]
        search_val_df = outer_train_df[outer_train_df[config.source_id_column].isin(search_val_ids)]
        print(f"[nested] rank search split: train={len(search_train_df)} rows | val={len(search_val_df)} rows")

        rank_performance = _load_search_checkpoint(output_dir, outer_fold_id) if resume else {}
        if rank_performance:
            print(f"[nested] resuming rank search, already have: {rank_performance}")

        for rank_candidate in config.rank_grid:
            if rank_candidate in rank_performance:
                print(f"[nested] rank={rank_candidate}: already evaluated (PR-AUC={rank_performance[rank_candidate]:.4f}), skipping.")
                continue

            rank_t0 = time.time()
            print(f"[nested] --- evaluating rank candidate {rank_candidate} ---")

            val_scores, tmp_model = train_lora_model_safe(
                search_train_df, search_val_df, tokenizer, rank=rank_candidate,
                epochs=config.search_epochs, batch_size=config.train_batch_size,
                grad_accum_steps=config.grad_accum_steps, eval_batch_size=config.eval_batch_size,
                num_workers=config.num_workers, device=device,
                hf_cache_dir=config.hf_cache_dir, code_column=config.code_column,
                max_length=config.max_length, log_prefix="    ",
            )
            prauc = float(average_precision_score(search_val_df[config.label_column].values, val_scores))
            rank_performance[rank_candidate] = prauc

            print(f"[nested] rank={rank_candidate} | PR-AUC={prauc:.4f} | {(time.time()-rank_t0)/60:.1f} min")

            _save_search_checkpoint(output_dir, outer_fold_id, rank_performance)

            del tmp_model
            gc.collect()
            torch.cuda.empty_cache()

        optimal_rank = max(rank_performance, key=rank_performance.get)
        print(f"[nested] Selected rank={optimal_rank} for outer fold {outer_fold_id} | scores={rank_performance}")

        print(f"[nested] --- final refit on full outer_train, rank={optimal_rank}, {config.epochs} epochs ---")
        refit_t0 = time.time()
        outer_val_scores, final_outer_model = train_lora_model_safe(
            outer_train_df, outer_val_df, tokenizer, rank=optimal_rank,
            epochs=config.epochs, batch_size=config.train_batch_size,
            grad_accum_steps=config.grad_accum_steps, eval_batch_size=config.eval_batch_size,
            num_workers=config.num_workers, device=device,
            hf_cache_dir=config.hf_cache_dir, code_column=config.code_column,
            max_length=config.max_length, log_prefix="    ",
        )
        print(f"[nested] refit done in {(time.time()-refit_t0)/60:.1f} min")

        fold_oof = pd.DataFrame({
            config.source_id_column: outer_val_df[config.source_id_column].values,
            config.project_column: outer_val_df[config.project_column].values,
            "label": outer_val_df[config.label_column].astype(int).values,
            "y_score": outer_val_scores,
            "fold": outer_fold_id,
        })

        selected_row = {"outer_fold_id": outer_fold_id, "selected_rank": optimal_rank, "search_scores": rank_performance}
        training_row = {
            "outer_fold_id": outer_fold_id,
            "selected_rank": optimal_rank,
            "n_train": int(len(outer_train_df)),
            "n_val": int(len(outer_val_df)),
            "elapsed_minutes": (time.time() - fold_t0) / 60,
        }

        if outer_fold_id == fold_ids[-1]:
            final_outer_model.save_pretrained(output_dir / "final_exp5_lora_adapter")
            print(f"[nested] saved final fold LoRA adapter to {output_dir / 'final_exp5_lora_adapter'}")

        _write_outer_checkpoint(output_dir, outer_fold_id, fold_oof, selected_row, training_row)

        oof_parts.append(fold_oof)
        selected_rows.append(selected_row)
        outer_training_rows.append(training_row)
        completed_folds.append(outer_fold_id)

        _update_run_state(state_path, completed_folds, status="running")

        del outer_train_df, outer_val_df, search_train_df, search_val_df, final_outer_model
        gc.collect()
        torch.cuda.empty_cache()

        print(f"[nested] Outer fold {outer_fold_id} done in {training_row['elapsed_minutes']:.1f} min | checkpoint saved | total elapsed {(time.time()-t0)/60:.1f} min")

    oof_predictions = pd.concat(oof_parts, axis=0).reset_index(drop=True)

    eval_config = EvaluationConfig(threshold=config.decision_threshold, expected_n_folds=len(fold_ids))
    eval_results = evaluation.evaluate_oof_predictions(oof_predictions, config=eval_config)

    selected_df = pd.DataFrame(selected_rows)
    outer_training_df = pd.DataFrame(outer_training_rows)

    artifacts = {
        "oof_predictions": output_dir / "exp5_nested_oof_predictions.parquet",
        "selected_rank_per_fold": output_dir / "exp5_selected_rank_per_outer_fold.csv",
        "outer_training_audit": output_dir / "exp5_outer_training_audit.csv",
        "run_metadata": output_dir / "exp5_nested_run_metadata.json",
    }
    oof_predictions.to_parquet(artifacts["oof_predictions"], index=False)
    selected_df.to_csv(artifacts["selected_rank_per_fold"], index=False)
    outer_training_df.to_csv(artifacts["outer_training_audit"], index=False)

    metadata = {
        "exp5_version": EXP5_VERSION,
        "config": {**asdict(config), "rank_grid": list(config.rank_grid)},
        "runtime_seconds": time.time() - t0,
        **(additional_metadata or {}),
    }
    with open(artifacts["run_metadata"], "w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2, default=str)

    _update_run_state(state_path, completed_folds, status="completed")

    print(f"\\n[nested] EXP-5 nested rank search complete in {(time.time()-t0)/60:.1f} min")

    return {
        "oof_predictions": oof_predictions,
        "evaluation": eval_results,
        "selected_rank": selected_df,
        "outer_fold_training": outer_training_df,
        "artifacts": artifacts,
        "tokenizer": tokenizer,
    }


def run_exp5_canonical_retrain(
    development_frame: pd.DataFrame,
    tokenizer,
    selected_rank: int,
    holdout_frame: pd.DataFrame,
    config: Exp5Config,
    output_dir: Path,
) -> Dict[str, Any]:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    print(f"[canonical] Retraining on full development set ({len(development_frame)} rows), rank={selected_rank}, {config.epochs} epochs")
    t0 = time.time()

    holdout_scores, global_model = train_lora_model_safe(
        development_frame, holdout_frame, tokenizer, rank=selected_rank,
        epochs=config.epochs, batch_size=config.train_batch_size,
        grad_accum_steps=config.grad_accum_steps, eval_batch_size=config.eval_batch_size,
        num_workers=config.num_workers, device=device,
        hf_cache_dir=config.hf_cache_dir, code_column=config.code_column,
        max_length=config.max_length, log_prefix="  ",
    )
    global_model.save_pretrained(output_dir / "final_canonical_lora_model")
    print(f"[canonical] Done in {(time.time()-t0)/60:.1f} min | model saved to {output_dir / 'final_canonical_lora_model'}")

    holdout_predictions = pd.DataFrame({
        config.source_id_column: holdout_frame[config.source_id_column].values,
        config.project_column: holdout_frame[config.project_column].values,
        "label": holdout_frame[config.label_column].astype(int).values,
        "y_score": holdout_scores,
        "fold": 0,
    })

    del global_model
    gc.collect()
    torch.cuda.empty_cache()

    return {"holdout_predictions": holdout_predictions}


def run_exp5_holdout_evaluation(
    holdout_predictions: pd.DataFrame,
    config: Exp5Config,
    output_dir: Path,
    n_bootstrap: int = 1000,
    confidence: float = 0.95,
) -> Dict[str, Any]:
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    holdout_predictions.to_csv(output_dir / "exp5_holdout_predictions.csv", index=False)

    eval_config = EvaluationConfig(threshold=config.decision_threshold, expected_n_folds=1)
    holdout_metrics = evaluation.evaluate_oof_predictions(holdout_predictions, config=eval_config)
    print(evaluation.format_metric_report(holdout_metrics["pooled_metrics"]))

    ci_result = bootstrap_metric_ci(
        holdout_predictions,
        metric="average_precision_pr_auc",
        group_column="project",
        n_bootstrap=n_bootstrap,
        confidence=confidence,
        random_state=config.random_state,
    )
    with open(output_dir / "exp5_holdout_pr_auc_bootstrap_ci.json", "w", encoding="utf-8") as f:
        json.dump(ci_result.as_dict(), f, indent=2)
    print(format_ci_report(ci_result))

    y_true = holdout_predictions["label"].values
    y_score = holdout_predictions["y_score"].values
    precision, recall, _ = precision_recall_curve(y_true, y_score)
    ap = holdout_metrics["pooled_metrics"]["average_precision_pr_auc"]
    plt.figure(figsize=(6, 5))
    plt.plot(recall, precision, color="b", label=f"EXP-5 LoRA (PR-AUC = {ap:.4f})")
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve - Frozen Outer Holdout")
    plt.legend(loc="lower left")
    plt.grid(True)
    plt.savefig(output_dir / "exp5_outer_holdout_pr_curve.png")
    plt.close()

    y_pred = (y_score >= config.decision_threshold).astype(int)
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(4, 4))
    plt.imshow(cm, cmap=plt.cm.Blues)
    plt.title("Confusion Matrix - Frozen Outer Holdout")
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.xticks([0, 1], ["Non-Vuln (0)", "Vuln (1)"])
    plt.yticks([0, 1], ["Non-Vuln (0)", "Vuln (1)"])
    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i, j]), ha="center", va="center")
    plt.tight_layout()
    plt.savefig(output_dir / "exp5_outer_holdout_confusion_matrix.png")
    plt.close()

    return {
        "holdout_metrics": holdout_metrics,
        "bootstrap_ci": ci_result.as_dict(),
        "y_pred": y_pred,
    }
''')
print("Wrote", "case_study_2/exp5/exp5_nested_rank.py")

Wrote case_study_2/exp5/exp5_nested_rank.py


## 6. Import project modules


In [13]:
import sys

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith("case_study_2") or mod_name.startswith("case_study_1"):
        del sys.modules[mod_name]

from case_study_2.data_loader import create_dataloader, get_class_weights
from case_study_2.models import (
    configure_huggingface_cache,
    load_code_tokenizer,
    DEFAULT_CODE_MODEL,
    DEFAULT_CODE_TOKENIZER,
    count_trainable_parameters,
    CodeSequenceClassifier,
    infer_lora_target_modules,
)
from case_study_2.exp5.exp5_lora import train_lora_model, train_lora_model_safe
from case_study_2.exp5.exp5_nested_rank import (
    Exp5Config,
    run_exp5_nested_rank,
    run_exp5_canonical_retrain,
    run_exp5_holdout_evaluation,
)
from case_study_1.confidence_intervals import paired_bootstrap_metric_ci, format_paired_ci_report

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 7. Load dataset and frozen manifests


In [14]:
import pandas as pd

full_df = pd.read_parquet(NORMALIZED_PARQUET)
outer_manifest_df = pd.read_parquet(OUTER_MANIFEST_PATH)
inner_manifest_df = pd.read_parquet(INNER_MANIFEST_PATH)

print("full_df rows:", len(full_df))
print("outer_manifest_df rows:", len(outer_manifest_df))
print("inner_manifest_df rows:", len(inner_manifest_df))

full_df rows: 261667
outer_manifest_df rows: 261667
inner_manifest_df rows: 203958


## 8. Build development and holdout frames


In [15]:
required_columns = {"source_row_id", "normalized_code", "label", "project"}
missing_columns = required_columns - set(full_df.columns)
if missing_columns:
    raise ValueError(f"Missing columns in full_df: {missing_columns}")

full_indexed = full_df.set_index("source_row_id", drop=False)
dev_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "development", "source_row_id"].tolist())
holdout_ids = set(outer_manifest_df.loc[outer_manifest_df["partition"] == "outer_holdout", "source_row_id"].tolist())
inner_ids = set(inner_manifest_df["source_row_id"].tolist())

if dev_ids != inner_ids:
    raise RuntimeError("Development partition and inner manifest coverage do not match.")
if dev_ids.intersection(holdout_ids):
    raise RuntimeError("Development and holdout partitions overlap.")

development_frame = full_indexed.loc[full_indexed["source_row_id"].isin(dev_ids)].reset_index(drop=True)
holdout_frame = full_indexed.loc[full_indexed["source_row_id"].isin(holdout_ids)].reset_index(drop=True)

print("development_frame rows:", len(development_frame))
print("holdout_frame rows:", len(holdout_frame))

development_frame rows: 203958
holdout_frame rows: 57709


## 9. Configuration object


In [16]:
exp5_config = Exp5Config(
    hf_cache_dir=HF_CACHE_DIR,
    rank_grid=RANK_GRID,
    epochs=EPOCHS,
    search_epochs=SEARCH_EPOCHS,
    search_n_splits=SEARCH_N_SPLITS,
    train_batch_size=TRAIN_BATCH_SIZE,
    grad_accum_steps=GRAD_ACCUM_STEPS,
    num_workers=NUM_WORKERS,
    eval_batch_size=EVAL_BATCH_SIZE,
)
print(exp5_config)

Exp5Config(experiment_name='cs2_exp5_graphcodebert_lora', code_column='normalized_code', source_id_column='source_row_id', label_column='label', project_column='project', fold_column='fold', hf_cache_dir=PosixPath('/workspace/IntelligentSystemProject/hf_cache'), max_length=512, train_batch_size=32, grad_accum_steps=2, epochs=4, rank_grid=(8, 16, 32), inner_n_splits=3, inner_random_state=20260707, decision_threshold=0.5, search_epochs=2, search_n_splits=5, num_workers=8, eval_batch_size=64, n_splits=5, random_state=42, verbose=True)


## 10. Sanity-check LoRA target modules


In [17]:
configure_huggingface_cache(HF_CACHE_DIR)
_tok_check = load_code_tokenizer(DEFAULT_CODE_TOKENIZER, hf_cache_dir=HF_CACHE_DIR)

_probe_model = CodeSequenceClassifier(model_name=DEFAULT_CODE_MODEL, freeze_backbone=False, dtype_policy="bfloat16", hf_cache_dir=HF_CACHE_DIR)
_target_modules = infer_lora_target_modules(_probe_model)
print("Inferred LoRA target_modules:", _target_modules)

del _probe_model
torch.cuda.empty_cache()

2026-08-04 03:34:08.820533: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-08-04 03:34:08.832244: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1785814448.846696    4534 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1785814448.851518    4534 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-04 03:34:08.868375: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

Inferred LoRA target_modules: ['query', 'value']


## 11. Smoke test on a small subsample


In [18]:
import time
from sklearn.metrics import average_precision_score

if RUN_SMOKE_TEST:
    sample_df = development_frame.sample(n=min(2000, len(development_frame)), random_state=42).reset_index(drop=True)
    smoke_train = sample_df.iloc[:1500].reset_index(drop=True)
    smoke_val = sample_df.iloc[1500:].reset_index(drop=True)

    print(f"[smoke] train={len(smoke_train)} rows | val={len(smoke_val)} rows")

    t0 = time.time()
    smoke_scores, smoke_model = train_lora_model_safe(
        smoke_train, smoke_val, _tok_check, rank=RANK_GRID[0], epochs=1,
        batch_size=exp5_config.train_batch_size, grad_accum_steps=exp5_config.grad_accum_steps,
        eval_batch_size=exp5_config.eval_batch_size, num_workers=exp5_config.num_workers,
        device=DEVICE, hf_cache_dir=HF_CACHE_DIR, code_column=exp5_config.code_column,
        max_length=exp5_config.max_length,
    )
    smoke_prauc = float(average_precision_score(smoke_val[exp5_config.label_column].values, smoke_scores))
    print(f"[smoke] PR-AUC={smoke_prauc:.4f} | elapsed={(time.time()-t0)/60:.1f} min")

    del smoke_model
    torch.cuda.empty_cache()
else:
    print("RUN_SMOKE_TEST=False; skipping.")

[smoke] train=1500 rows | val=500 rows


Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[exp5-lora] rank=8 | train_rows=1500 | val_rows=500 | batch_size=32 | grad_accum=2 | steps/epoch=47 | trainable=294,912 (0.236%) | total=124,941,313
[exp5-lora] epoch 1/1 done | avg_loss=1.2946 | epoch_time=0.1 min | total_elapsed=0.1 min | peak_VRAM=0.00 GB
[exp5-lora] training done, scoring validation set (500 rows)...
[smoke] PR-AUC=0.0579 | elapsed=0.2 min


## 12. Official nested LoRA rank search


In [19]:
if RUN_NESTED_OFFICIAL:
    nested_results = run_exp5_nested_rank(
        development_frame=development_frame,
        development_manifest=inner_manifest_df,
        config=exp5_config,
        output_dir=EXP5_OUTPUT_DIR,
        resume=True,
    )
    
    selected_ranks = nested_results["selected_rank"]["selected_rank"].tolist()
    best_rank = int(nested_results["selected_rank"]["selected_rank"].mode()[0])
    print(f"\nOuter fold selected ranks: {selected_ranks} -> Majority chosen rank for retrain: {best_rank}")

if RUN_CANONICAL_RETRAIN and RUN_NESTED_OFFICIAL:
    canonical_results = run_exp5_canonical_retrain(
        development_frame=development_frame,
        tokenizer=nested_results["tokenizer"],
        selected_rank=best_rank,
        holdout_frame=holdout_frame,
        config=exp5_config,
        output_dir=EXP5_OUTPUT_DIR,
    )

if RUN_HOLDOUT_EVAL and RUN_CANONICAL_RETRAIN:
    eval_results = run_exp5_holdout_evaluation(
        holdout_predictions=canonical_results["holdout_predictions"],
        config=exp5_config,
        output_dir=EXP5_OUTPUT_DIR,
    )
    print("\nEXP-5 holdout evaluation complete!")

[nested] Starting EXP-5 nested rank search over 5 outer folds, rank_grid=(8, 16, 32)
[nested] Search phase: 2 epoch(s), single held-out split | Refit phase: 4 epoch(s), full training
[nested] Outer fold 0: loaded from checkpoint, skipping.
[nested] Outer fold 1: loaded from checkpoint, skipping.
[nested] Outer fold 2: loaded from checkpoint, skipping.
[nested] Outer fold 3: loaded from checkpoint, skipping.
[nested] Outer fold 4: loaded from checkpoint, skipping.

[nested] EXP-5 nested rank search complete in 0.1 min

Outer fold selected ranks: [16, 16, 8, 32, 8] -> Majority chosen rank for retrain: 8
[canonical] Retraining on full development set (203958 rows), rank=8, 4 epochs


Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  [exp5-lora] rank=8 | train_rows=203958 | val_rows=57709 | batch_size=32 | grad_accum=2 | steps/epoch=6374 | trainable=294,912 (0.236%) | total=124,941,313
  [exp5-lora] VRAM after model load: 0.28 GB
  [exp5-lora] epoch 1/4 step 50/6374 | avg_loss_so_far=1.4487 | elapsed=0.1 min | ETA epoch ~11.2 min
  [exp5-lora] epoch 1/4 step 100/6374 | avg_loss_so_far=1.3683 | elapsed=0.2 min | ETA epoch ~10.6 min
  [exp5-lora] epoch 1/4 step 150/6374 | avg_loss_so_far=1.3286 | elapsed=0.2 min | ETA epoch ~10.4 min
  [exp5-lora] epoch 1/4 step 200/6374 | avg_loss_so_far=1.3006 | elapsed=0.3 min | ETA epoch ~10.2 min
  [exp5-lora] epoch 1/4 step 250/6374 | avg_loss_so_far=1.2553 | elapsed=0.4 min | ETA epoch ~10.1 min
  [exp5-lora] epoch 1/4 step 300/6374 | avg_loss_so_far=1.2610 | elapsed=0.5 min | ETA epoch ~10.0 min
  [exp5-lora] epoch 1/4 step 350/6374 | avg_loss_so_far=1.2516 | elapsed=0.6 min | ETA epoch ~9.8 min
  [exp5-lora] epoch 1/4 step 400/6374 | avg_loss_so_far=1.2420 | elapsed=0.7 mi

## 13. Canonical retrain and frozen outer holdout scoring


In [21]:
if RUN_CANONICAL_RETRAIN and nested_results is not None:
    global_selected_rank = int(nested_results["selected_rank"]["selected_rank"].mode()[0])
    retrain_results = run_exp5_canonical_retrain(
        development_frame=development_frame,
        tokenizer=nested_results["tokenizer"], 
        selected_rank=global_selected_rank,
        holdout_frame=holdout_frame,
        config=exp5_config,
        output_dir=EXP5_OUTPUT_DIR,
    )
    print("Canonical model trained with rank =", global_selected_rank)
else:
    retrain_results = None
    print("RUN_CANONICAL_RETRAIN=False or no nested results; skipping.")


[canonical] Retraining on full development set (203958 rows), rank=8, 4 epochs


Some weights of RobertaModel were not initialized from the model checkpoint at microsoft/graphcodebert-base and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  [exp5-lora] rank=8 | train_rows=203958 | val_rows=57709 | batch_size=32 | grad_accum=2 | steps/epoch=6374 | trainable=294,912 (0.236%) | total=124,941,313
  [exp5-lora] VRAM after model load: 0.28 GB
  [exp5-lora] epoch 1/4 step 50/6374 | avg_loss_so_far=1.3156 | elapsed=0.1 min | ETA epoch ~11.4 min
  [exp5-lora] epoch 1/4 step 100/6374 | avg_loss_so_far=1.2767 | elapsed=0.2 min | ETA epoch ~10.7 min
  [exp5-lora] epoch 1/4 step 150/6374 | avg_loss_so_far=1.2751 | elapsed=0.3 min | ETA epoch ~10.4 min
  [exp5-lora] epoch 1/4 step 200/6374 | avg_loss_so_far=1.2773 | elapsed=0.3 min | ETA epoch ~10.2 min
  [exp5-lora] epoch 1/4 step 250/6374 | avg_loss_so_far=1.2651 | elapsed=0.4 min | ETA epoch ~10.1 min
  [exp5-lora] epoch 1/4 step 300/6374 | avg_loss_so_far=1.2389 | elapsed=0.5 min | ETA epoch ~10.0 min
  [exp5-lora] epoch 1/4 step 350/6374 | avg_loss_so_far=1.2233 | elapsed=0.6 min | ETA epoch ~9.9 min
  [exp5-lora] epoch 1/4 step 400/6374 | avg_loss_so_far=1.2128 | elapsed=0.7 mi

## 14. Frozen outer holdout evaluation with bootstrap confidence interval


In [22]:
if RUN_HOLDOUT_EVAL and retrain_results is not None:
    holdout_results = run_exp5_holdout_evaluation(
        holdout_predictions=retrain_results["holdout_predictions"],
        config=exp5_config,
        output_dir=EXP5_OUTPUT_DIR,
    )
else:
    holdout_results = None
    print("RUN_HOLDOUT_EVAL=False or no canonical retrain results; skipping.")


Pooled Out-of-Fold Evaluation
                   n_samples: 57709
                vulnerable_1: 3211
            non_vulnerable_0: 54498
               positive_rate: 0.055641
                   threshold: 0.500000
    average_precision_pr_auc: 0.131495
                   precision: 0.091567
                      recall: 0.693553
                          f1: 0.161775
                         mcc: 0.133762
                 specificity: 0.594591
         false_positive_rate: 0.405409
               true_negative: 32404
              false_positive: 22094
              false_negative: 984
               true_positive: 2227
average_precision_pr_auc: point estimate = 0.1315
  95% CI (project-block bootstrap): [0.1044, 0.1680]
  valid resamples: 1000/1000 (0 degenerate, dropped)
  n_projects: 203, random_state=42
  Reflects sampling variability within this dataset only; not an estimate of generalization to C functions outside this collection.


## 15. Paired comparison against EXP-3 on the frozen holdout


In [23]:
EXP3_HOLDOUT_PREDICTIONS_PATH = OUTPUT_ROOT / "case_study_2" / "exp3_codeberta_linear_probe_v1" / "exp3_holdout_predictions.csv"

if RUN_HOLDOUT_EVAL and retrain_results is not None and EXP3_HOLDOUT_PREDICTIONS_PATH.exists():
    exp3_holdout = pd.read_csv(EXP3_HOLDOUT_PREDICTIONS_PATH)
    comparison = paired_bootstrap_metric_ci(
        predictions_a=retrain_results["holdout_predictions"],
        predictions_b=exp3_holdout,
        experiment_name_a="EXP-5 HEFT",
        experiment_name_b="EXP-3 linear probe",
        metric="average_precision_pr_auc",
    )
    print(format_paired_ci_report(comparison))
else:
    print("Run EXP-5 holdout evaluation and the EXP-3 notebook first.")


Run EXP-5 holdout evaluation and the EXP-3 notebook first.


## 16. Cleanup


In [24]:
import gc
import shutil
import torch

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("VRAM allocated:", torch.cuda.memory_allocated() / 1e9, "GB")

total, used, free = shutil.disk_usage(WORKSPACE_ROOT)
print(f"Disk usage at {WORKSPACE_ROOT}: {used/1e9:.1f} GB used / {total/1e9:.1f} GB total ({free/1e9:.1f} GB free)")
if used / 1e9 > STORAGE_CAP_GB:
    print(f"WARNING: workspace usage exceeds the {STORAGE_CAP_GB} GB storage cap -- consider pruning old checkpoints under {EXP5_OUTPUT_DIR}.")


VRAM allocated: 0.01703936 GB
Disk usage at /workspace: 5129.5 GB used / 5714.2 GB total (296.6 GB free)
